# Submission — Tuned LightGBM (Optuna)

Loads the LightGBM booster trained locally with Optuna-tuned hyperparameters and DTW features. Single-model submission — no ensemble.

**Inputs (attach as Kaggle datasets):**
- `rogii-wellbore-geology-prediction` (the competition data)
- A private dataset bundling our `src/` + final LGB artefact. Adjust `BUNDLE` below.

**Bundle layout expected:**
```
<bundle>/
  src/baseline.py
  artefacts/features.json
  artefacts/lgb_optuna/best_params.json
  artefacts/lgb_optuna/final_lgb.txt
```

**Local OOF reference:** LGB RMSE 12.640 on tag-grouped 5-fold CV. Tuned on a 30%-group subsample with 3-fold; best params: lr=0.036, num_leaves=36, min_data_in_leaf=110, max_depth=2 (heavy regularisation, shallow trees).

In [ ]:
import os, sys, shutil
from pathlib import Path

# ── Paths — edit these to match your dataset slugs ───────────────────────
BUNDLE = Path('/kaggle/input/wellbore-baseline-bundle')
COMP   = Path('/kaggle/input/rogii-wellbore-geology-prediction')
WORK   = Path('/kaggle/working')

assert BUNDLE.exists(), f'Missing bundle dataset: {BUNDLE}'
assert COMP.exists(),   f'Missing competition input: {COMP}'

ART_SRC = BUNDLE / 'artefacts'
ART_DST = WORK   / 'artefacts'
if not ART_DST.exists():
    shutil.copytree(ART_SRC, ART_DST)

os.environ['ARTEFACT_DIR'] = str(ART_DST)
os.environ['OUTPUT_DIR']   = str(WORK / 'outputs')
os.environ['DATA_DIR']     = str(COMP)

sys.path.insert(0, str(BUNDLE / 'src'))
print('Artefacts:', ART_DST)
print('LGB    :', ART_DST / 'lgb_optuna' / 'final_lgb.txt',
      '(exists:', (ART_DST / 'lgb_optuna' / 'final_lgb.txt').exists(), ')')

In [ ]:
import baseline as bl
import lightgbm as lgb
import numpy as np, pandas as pd

n_test = len(list(bl.TEST_DIR.glob('*__horizontal_well.csv')))
print(f'TEST_DIR: {bl.TEST_DIR}  test wells: {n_test}')

## Build test features

Same FE pipeline as the ensemble notebook (DTW included). If you've already run the ensemble notebook in this session, you can skip the rebuild and load `WORK / 'test_df.parquet'` instead.

In [ ]:
test_df_path = WORK / 'test_df.parquet'
if test_df_path.exists():
    print('Loading cached test_df from current session …')
    test_df = pd.read_parquet(test_df_path)
else:
    test_df = bl.build_dataset(bl.TEST_DIR, is_train=False)
    test_df.to_parquet(test_df_path, index=False)

feature_cols = bl.am.load_json(bl.am.FEATURES_LIST)
missing = [c for c in feature_cols if c not in test_df.columns]
print(f'test_df: {test_df.shape}  features: {len(feature_cols)}  missing: {len(missing)}')
if missing:
    raise RuntimeError(f'test_df is missing {len(missing)} feature columns: {missing[:10]}')

## Load LGB booster, predict

In [ ]:
booster_path = ART_DST / 'lgb_optuna' / 'final_lgb.txt'
booster = lgb.Booster(model_file=str(booster_path))

best_params = bl.am.load_json('lgb_optuna/best_params')  # reference only
print('best_params used at training:')
for k, v in best_params.items():
    print(f'  {k:>25s} = {v}')
print(f'\nbooster: {booster.num_trees()} trees, {booster.num_feature()} features')

In [ ]:
delta_pred = booster.predict(test_df[feature_cols].to_numpy()).astype(np.float64)
abs_tvt    = delta_pred + test_df['last_known_tvt'].to_numpy()
print(f'delta range: {delta_pred.min():.3f}  {delta_pred.mean():+.3f}  {delta_pred.max():.3f}')
print(f'abs TVT   range: {abs_tvt.min():.3f}  {abs_tvt.mean():+.3f}  {abs_tvt.max():.3f}')

## Build submission

In [ ]:
sample = pd.read_csv(COMP / 'sample_submission.csv')
out    = WORK / 'submission.csv'
sub    = bl.build_submission(test_df, abs_tvt, sample, out)
print(sub.head())
print(f'\nrows: {len(sub):,}  rows-without-prediction: {sub["tvt"].isna().sum()}')
print(f'tvt summary:\n{sub["tvt"].describe()}')